In [ ]:
import argparse
import sqlite3
from datetime import date

import pandas as pd
import os


def get_or_create(cur, table, unique_cols, values, id_col):
    where = " AND ".join(f"{c} = ?" for c in unique_cols)
    cur.execute(f"SELECT {id_col} FROM {table} WHERE {where}", values)
    row = cur.fetchone()
    if row:
        return row[0]
    cols = ", ".join(unique_cols)
    placeholders = ", ".join("?" for _ in unique_cols)
    cur.execute(f"INSERT INTO {table} ({cols}) VALUES ({placeholders})", values)
    return cur.lastrowid


def load(input_csv, db_path, schema_sql):
    df = pd.read_csv(input_csv)
    conn = sqlite3.connect(db_path)
    cur = conn.cursor()
    cur.executescript(open(schema_sql).read())

    for _, r in df.iterrows():
        y, m, d = map(int, r["arrival_date_iso"].split("-"))
        weekday = date(y, m, d).weekday()
        date_id = get_or_create(
            cur, "Dim_Date", ["full_date", "day", "month", "quarter", "year", "is_weekend"],
            [r["arrival_date_iso"], d, m, (m - 1) // 3 + 1, y, int(weekday >= 5)], "date_id",
        )
        market_id = get_or_create(
            cur, "Dim_Market", ["market_name", "district", "state"],
            [r["market"], r["district"], r["state"]], "market_id",
        )
        commodity_id = get_or_create(
            cur, "Dim_Commodity", ["commodity_name"], [r["commodity_std"]], "commodity_id",
        )
        variety_id = get_or_create(
            cur, "Dim_Variety", ["variety_name", "grade"],
            [r["variety_std"], r.get("grade")], "variety_id",
        )
        cur.execute(
            """INSERT INTO Fact_MandiPrice
               (date_id, market_id, commodity_id, variety_id,
                min_price, max_price, modal_price, is_estimated)
               VALUES (?, ?, ?, ?, ?, ?, ?, ?)""",
            (date_id, market_id, commodity_id, variety_id,
             r["min_price"], r["max_price"], r["modal_price"], int(r["is_estimated"])),
        )

    conn.commit()

    counts = {}
    for t in ["Dim_Date", "Dim_Market", "Dim_Commodity", "Dim_Variety", "Fact_MandiPrice"]:
        counts[t] = cur.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
    conn.close()
    return counts


if __name__ == "__main__":
    os.makedirs("out", exist_ok=True)
    os.makedirs("sql", exist_ok=True)
    if not os.path.exists("out/clean_prices.csv"):
        with open("out/clean_prices.csv", "w") as f:
            f.write("arrival_date_iso,market,district,state,commodity_std,variety_std,grade,min_price,max_price,modal_price,is_estimated\n")
            # Add sample data to the CSV
            f.write("2023-01-01,MarketA,DistrictX,StateY,Tomato,Red,GradeA,10.0,15.0,12.0,0\n")
            f.write("2023-01-01,MarketB,DistrictY,StateZ,Potato,White,GradeB,5.0,8.0,6.0,0\n")
            f.write("2023-01-02,MarketA,DistrictX,StateY,Tomato,Green,GradeA,11.0,16.0,13.0,0\n")
    with open("sql/schema.sql", "w") as f:
        f.write("""-- Define the schema for the data warehouse

CREATE TABLE IF NOT EXISTS Dim_Date (
    date_id INTEGER PRIMARY KEY AUTOINCREMENT,
    full_date TEXT UNIQUE NOT NULL,
    day INTEGER NOT NULL,
    month INTEGER NOT NULL,
    quarter INTEGER NOT NULL,
    year INTEGER NOT NULL,
    is_weekend INTEGER NOT NULL
);

CREATE TABLE IF NOT EXISTS Dim_Market (
    market_id INTEGER PRIMARY KEY AUTOINCREMENT,
    market_name TEXT UNIQUE NOT NULL,
    district TEXT NOT NULL,
    state TEXT NOT NULL
);

CREATE TABLE IF NOT EXISTS Dim_Commodity (
    commodity_id INTEGER PRIMARY KEY AUTOINCREMENT,
    commodity_name TEXT UNIQUE NOT NULL
);

CREATE TABLE IF NOT EXISTS Dim_Variety (
    variety_id INTEGER PRIMARY KEY AUTOINCREMENT,
    variety_name TEXT UNIQUE NOT NULL,
    grade TEXT
);

CREATE TABLE IF NOT EXISTS Fact_MandiPrice (
    fact_id INTEGER PRIMARY KEY AUTOINCREMENT,
    date_id INTEGER NOT NULL,
    market_id INTEGER NOT NULL,
    commodity_id INTEGER NOT NULL,
    variety_id INTEGER NOT NULL,
    min_price REAL NOT NULL,
    max_price REAL NOT NULL,
    modal_price REAL NOT NULL,
    is_estimated INTEGER NOT NULL,
    FOREIGN KEY (date_id) REFERENCES Dim_Date(date_id),
    FOREIGN KEY (market_id) REFERENCES Dim_Market(market_id),
    FOREIGN KEY (commodity_id) REFERENCES Dim_Commodity(commodity_id),
    FOREIGN KEY (variety_id) REFERENCES Dim_Variety(variety_id)
);
""")

    ap = argparse.ArgumentParser()
    ap.add_argument("--input", default="out/clean_prices.csv")
    ap.add_argument("--db", default="out/mandipulse.db")
    ap.add_argument("--schema", default="sql/schema.sql")
    args, unknown = ap.parse_known_args()

    counts = load(args.input, args.db, args.schema)
    print(f"Loaded warehouse at {args.db}:")
    for t, c in counts.items():
        print(f"  {t}: {c} rows")

Loaded warehouse at out/mandipulse.db:
  Dim_Date: 2 rows
  Dim_Market: 2 rows
  Dim_Commodity: 2 rows
  Dim_Variety: 3 rows
  Fact_MandiPrice: 3 rows
